# EDA and Data Cleaning

UCI Default of Credit Card Clients dataset. 30,000 rows, demographic info,
payment history, bill amounts, and default outcome.

Known quirks to check and document:
- `EDUCATION` and `MARRIAGE` have undocumented category codes (0, 5, 6 in
  EDUCATION; 0 in MARRIAGE) not covered by the official data dictionary.
- `PAY_0`..`PAY_6` repayment status codes include -2 and 0 which are not
  documented in the original UCI description.
- Column is literally named `PAY_0` instead of `PAY_1` (off-by-one naming
  inconsistency in the source data).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/default_credit_card.csv')
df = df.rename(columns={'default payment next month': 'DEFAULT'})
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## EDUCATION and MARRIAGE cleanup

Documented codes:
- EDUCATION: 1 graduate school, 2 university, 3 high school, 4 others
- MARRIAGE: 1 married, 2 single, 3 others

But the data contains extra undocumented codes.

In [ ]:
print(df['EDUCATION'].value_counts())
print()
print(df['MARRIAGE'].value_counts())

In [ ]:
# Fold undocumented EDUCATION codes (0, 5, 6) into the existing 'others' bucket (4)
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# Fold undocumented MARRIAGE code (0) into the existing 'others' bucket (3)
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print(df['EDUCATION'].value_counts())
print()
print(df['MARRIAGE'].value_counts())

## PAY_0..PAY_6 repayment status cleanup

Documented codes: -1 pay duly, 1-9 months delayed.
Undocumented codes present in the data: -2 and 0. Based on how the dataset is
commonly interpreted, -2 means no consumption / no balance to pay, and 0 means
the balance was paid using revolving credit (not officially late, not fully
settled either). Both are kept as distinct values rather than merged into -1,
since collapsing them would erase a real distinction in repayment behavior.

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for c in pay_cols:
    print(c, sorted(df[c].unique()))

## Class balance

In [ ]:
df['DEFAULT'].value_counts(normalize=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x='DEFAULT', ax=axes[0])
sns.histplot(data=df, x='AGE', hue='DEFAULT', kde=True, ax=axes[1])
sns.histplot(data=df, x='LIMIT_BAL', hue='DEFAULT', kde=True, ax=axes[2])
plt.tight_layout()

In [ ]:
df.to_csv('../data/default_credit_card_clean.csv', index=False)
print('saved cleaned dataset')